## 06 Final Demo Checks

This notebook operationalizes the final overlap-detection configuration selected in Notebooks 03 and 04 and prepares the proof-of-concept demo artifacts.

### Main goal

The purpose of Notebook 05 is to verify that the final selected configuration works cleanly as an end-to-end demo setup and to export the final artifacts required by the existing PoC application.

It answers:

> Is the final selected overlap-detection configuration ready for practical proof-of-concept use?

### What this notebook does

- loads the final cleaned course dataset from Notebook 01
- rebuilds the final selected embedding input surface
- generates final embeddings for all courses using the selected model
- computes full pairwise cosine similarity across the course set
- applies the locked threshold to identify overlap candidates
- saves app-ready artifacts for the final Streamlit demo
- runs practical sanity checks on the final retrieval behavior

### What this notebook does not do

- it does not rerun the benchmark competition from Notebook 03
- it does not retune thresholds from Notebook 04
- it does not perform large-scale production deployment
- it does not replace a future pgvector-backed implementation

### Relationship to the final demo app

This notebook prepares and verifies the final artifacts that the Streamlit application can use.

In other words:

- **Notebook 05** = final system preparation and sanity checking
- **`app/streamlit_app.py`** = final interactive user-facing demo

In [10]:
from pathlib import Path
import os
import time
import warnings
from itertools import combinations

import numpy as np
import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI
from sklearn.preprocessing import normalize

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 160)

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"

THRESHOLD_TABLES_DIR = REPORTS_DIR / "tables" / "thresholds"
FINAL_DEMO_TABLES_DIR = REPORTS_DIR / "tables" / "final_demo"
FINAL_DEMO_FIGURES_DIR = REPORTS_DIR / "figures" / "final_demo"

ENV_PATH = PROJECT_ROOT / ".env"

COURSES_CSV_PATH = DATA_DIR / "courses_clean.csv"
FINAL_RECOMMENDATION_CSV_PATH = THRESHOLD_TABLES_DIR / "final_threshold_recommendation.csv"

FINAL_EMBEDDINGS_PARQUET_PATH = DATA_DIR / "final_embeddings.parquet"
FINAL_PAIRWISE_SIMILARITY_PARQUET_PATH = DATA_DIR / "final_pairwise_similarity.parquet"
FINAL_PAIRWISE_SIMILARITY_CSV_PATH = DATA_DIR / "final_pairwise_similarity.csv"
PGVECTOR_READY_PARQUET_PATH = DATA_DIR / "pgvector_ready_embeddings.parquet"
PGVECTOR_READY_CSV_PATH = DATA_DIR / "pgvector_ready_embeddings.csv"

FINAL_OVERLAP_CANDIDATES_CSV_PATH = FINAL_DEMO_TABLES_DIR / "final_demo_overlap_candidates.csv"
FINAL_TOP_MATCHES_CSV_PATH = FINAL_DEMO_TABLES_DIR / "final_demo_top_matches_per_course.csv"
FINAL_SYSTEM_SUMMARY_CSV_PATH = FINAL_DEMO_TABLES_DIR / "final_demo_system_summary.csv"
FINAL_CONFIGURATION_CSV_PATH = FINAL_DEMO_TABLES_DIR / "final_demo_selected_configuration.csv"

for path in [FINAL_DEMO_TABLES_DIR, FINAL_DEMO_FIGURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Courses CSV:", COURSES_CSV_PATH)
print("Final recommendation CSV:", FINAL_RECOMMENDATION_CSV_PATH)
print("Final demo tables dir:", FINAL_DEMO_TABLES_DIR)
print("Environment file:", ENV_PATH)

Project root: C:\Users\user\Documents\thesis
Courses CSV: C:\Users\user\Documents\thesis\data\courses_clean.csv
Final recommendation CSV: C:\Users\user\Documents\thesis\reports\tables\thresholds\final_threshold_recommendation.csv
Final demo tables dir: C:\Users\user\Documents\thesis\reports\tables\final_demo
Environment file: C:\Users\user\Documents\thesis\.env


In [11]:
load_dotenv(ENV_PATH)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()

if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY was not found in the .env file. "
        "Notebook 05 requires the selected OpenAI embedding model."
    )

openai_client = OpenAI(api_key=OPENAI_API_KEY)

print("OpenAI API key loaded: Yes")
print("OpenAI client initialized successfully.")

OpenAI API key loaded: Yes
OpenAI client initialized successfully.


In [12]:
# Cell 03 - Load cleaned courses and final recommendation

if not COURSES_CSV_PATH.exists():
    raise FileNotFoundError(f"courses_clean.csv not found: {COURSES_CSV_PATH}")

if not FINAL_RECOMMENDATION_CSV_PATH.exists():
    raise FileNotFoundError(f"Final recommendation CSV not found: {FINAL_RECOMMENDATION_CSV_PATH}")

courses_df = pd.read_csv(COURSES_CSV_PATH, encoding="utf-8", keep_default_na=False)
final_recommendation_df = pd.read_csv(FINAL_RECOMMENDATION_CSV_PATH, encoding="utf-8", keep_default_na=False)

print("courses_df shape:", courses_df.shape)
print("final_recommendation_df shape:", final_recommendation_df.shape)

display(final_recommendation_df)

courses_df shape: (50, 26)
final_recommendation_df shape: (2, 17)


,recommendation_type,model_name,text_variant,similarity_metric,selection_rule,locked_threshold,shortlist_rank,average_precision,roc_auc,dev_f1,dev_f05,test_f1,test_f05,test_precision,test_recall,dev_predicted_positive_count,test_predicted_positive_count
0,primary_recommendation,openai_text-embedding-3-large,enriched,cosine,max_f1,0.673784,1,0.939595,0.994870,0.894737,0.923913,0.769231,0.892857,1.0,0.625,18,5
1,conservative_alternative,openai_text-embedding-3-large,main,cosine,max_f1,0.659979,2,0.927445,0.995913,0.871795,0.885417,0.769231,0.892857,1.0,0.625,19,5


In [13]:
# Cell 04 - Lock the final primary configuration from Notebook 04

required_recommendation_columns = [
    "recommendation_type",
    "model_name",
    "text_variant",
    "similarity_metric",
    "selection_rule",
    "locked_threshold",
]

missing_recommendation_columns = [
    col for col in required_recommendation_columns
    if col not in final_recommendation_df.columns
]

if missing_recommendation_columns:
    raise ValueError(
        f"Missing required columns in final recommendation CSV: {missing_recommendation_columns}"
    )

primary_match_df = final_recommendation_df[
    final_recommendation_df["recommendation_type"] == "primary_recommendation"
].copy()

if primary_match_df.empty:
    raise ValueError(
        "No row with recommendation_type='primary_recommendation' found in final recommendation CSV."
    )

if len(primary_match_df) != 1:
    raise ValueError("Expected exactly one primary_recommendation row.")

primary_row = primary_match_df.iloc[0]

FINAL_MODEL_NAME = str(primary_row["model_name"])
FINAL_TEXT_VARIANT_NAME = str(primary_row["text_variant"])
FINAL_SIMILARITY_METRIC = str(primary_row["similarity_metric"])
FINAL_SELECTION_RULE = str(primary_row["selection_rule"])
FINAL_LOCKED_THRESHOLD = float(primary_row["locked_threshold"])

if FINAL_MODEL_NAME != "openai_text-embedding-3-large":
    raise ValueError(
        f"Notebook 05 currently supports the final OpenAI setup only, but got: {FINAL_MODEL_NAME}"
    )

FINAL_OPENAI_MODEL_ID = "text-embedding-3-large"

TEXT_VARIANT_COLUMN_MAP = {
    "main": "embedding_text_main",
    "enriched": "embedding_text_enriched",
    "full": "embedding_text_full",
}

if FINAL_TEXT_VARIANT_NAME not in TEXT_VARIANT_COLUMN_MAP:
    raise ValueError(f"Unexpected final text variant: {FINAL_TEXT_VARIANT_NAME}")

FINAL_TEXT_COLUMN = TEXT_VARIANT_COLUMN_MAP[FINAL_TEXT_VARIANT_NAME]

print("Final selected configuration loaded from Notebook 04:")
print(f"  Model           : {FINAL_MODEL_NAME}")
print(f"  Text variant    : {FINAL_TEXT_VARIANT_NAME}")
print(f"  Text column     : {FINAL_TEXT_COLUMN}")
print(f"  Similarity      : {FINAL_SIMILARITY_METRIC}")
print(f"  Selection rule  : {FINAL_SELECTION_RULE}")
print(f"  Locked threshold: {FINAL_LOCKED_THRESHOLD:.6f}")

Final selected configuration loaded from Notebook 04:
  Model           : openai_text-embedding-3-large
  Text variant    : enriched
  Text column     : embedding_text_enriched
  Similarity      : cosine
  Selection rule  : max_f1
  Locked threshold: 0.673784


In [14]:
# Cell 05 - Validate the course dataset for final demo preparation

required_course_columns = [
    "course_unit_code",
    "course_name",
    FINAL_TEXT_COLUMN,
]

missing_course_columns = [
    col for col in required_course_columns
    if col not in courses_df.columns
]

if missing_course_columns:
    raise ValueError(f"Missing required course columns: {missing_course_columns}")

if courses_df["course_unit_code"].duplicated().any():
    raise ValueError("Duplicate course_unit_code values found in courses_df.")

empty_text_count = (
    courses_df[FINAL_TEXT_COLUMN].fillna("").astype(str).str.strip() == ""
).sum()

if empty_text_count > 0:
    raise ValueError(
        f"Found {empty_text_count} empty rows in {FINAL_TEXT_COLUMN}. "
        "Notebook 05 requires a complete final input surface."
    )

print("All required course columns are present.")
print("Unique course codes:", courses_df["course_unit_code"].nunique())
print(f"Empty rows in {FINAL_TEXT_COLUMN}:", int(empty_text_count))

All required course columns are present.
Unique course codes: 50
Empty rows in embedding_text_enriched: 0


In [15]:
# Cell 06 - Build final course text dataframe

final_course_text_df = courses_df[[
    "course_unit_code",
    "course_name",
    FINAL_TEXT_COLUMN,
]].copy()

final_course_text_df = final_course_text_df.rename(columns={
    FINAL_TEXT_COLUMN: "final_embedding_text"
})

final_course_text_df["final_embedding_text"] = (
    final_course_text_df["final_embedding_text"]
    .astype(str)
    .str.replace("\r\n", "\n", regex=False)
    .str.replace("\r", "\n", regex=False)
    .str.strip()
)

print("final_course_text_df shape:", final_course_text_df.shape)
display(final_course_text_df.head(3))

final_course_text_df shape: (50, 3)


,course_unit_code,course_name,final_embedding_text
0,TT00CD70,Data Networks,Data Networks\nYou learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data ne...
1,TT00CD71,Linux Basics,"Linux Basics\nAfter completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work wit..."
2,TT00CD72,Servers and containers,Servers and containers\nYou understand the fundamental types and functions of servers. You understand the key concepts of various services offered by cloud ...


In [16]:
# Cell 07 - Helper functions

def normalize_structured_text(text: str) -> str:
    text = str(text).replace("\r\n", "\n").replace("\r", "\n")
    lines = [line.strip() for line in text.split("\n")]
    lines = [line for line in lines if line]
    return "\n".join(lines)

def batched(iterable, batch_size: int):
    for start in range(0, len(iterable), batch_size):
        yield iterable[start:start + batch_size]

def l2_normalize_embeddings(matrix: np.ndarray) -> np.ndarray:
    return normalize(matrix, norm="l2")

def cosine_similarity_from_normalized(vec1: np.ndarray, vec2: np.ndarray) -> float:
    return float(np.dot(vec1, vec2))

def encode_with_openai(texts, model_name, client, batch_size=64):
    all_vectors = []

    for batch_texts in batched(texts, batch_size=batch_size):
        response = client.embeddings.create(
            model=model_name,
            input=batch_texts,
        )
        batch_vectors = [item.embedding for item in response.data]
        all_vectors.extend(batch_vectors)

    raw_matrix = np.array(all_vectors, dtype=np.float32)
    normalized_matrix = l2_normalize_embeddings(raw_matrix)
    return raw_matrix, normalized_matrix

In [17]:
# Cell 08 - Generate final embeddings for all courses

final_texts = final_course_text_df["final_embedding_text"].apply(normalize_structured_text).tolist()

embedding_start_time = time.time()

raw_embedding_matrix, normalized_embedding_matrix = encode_with_openai(
    texts=final_texts,
    model_name=FINAL_OPENAI_MODEL_ID,
    client=openai_client,
    batch_size=64,
)

embedding_elapsed_seconds = time.time() - embedding_start_time

if raw_embedding_matrix.shape[0] != len(final_course_text_df):
    raise ValueError("Raw embedding row count does not match course count.")

if normalized_embedding_matrix.shape[0] != len(final_course_text_df):
    raise ValueError("Normalized embedding row count does not match course count.")

print("Raw embedding matrix shape:", raw_embedding_matrix.shape)
print("Normalized embedding matrix shape:", normalized_embedding_matrix.shape)
print(f"Embedding generation time: {embedding_elapsed_seconds:.2f} seconds")

Raw embedding matrix shape: (50, 3072)
Normalized embedding matrix shape: (50, 3072)
Embedding generation time: 1.45 seconds


In [18]:
# Cell 09 - Save the final embedding artifact

raw_embedding_norms = np.linalg.norm(raw_embedding_matrix, axis=1)
normalized_embedding_norms = np.linalg.norm(normalized_embedding_matrix, axis=1)

final_embeddings_df = final_course_text_df.copy()

final_embeddings_df["final_model_name"] = FINAL_MODEL_NAME
final_embeddings_df["final_openai_model_id"] = FINAL_OPENAI_MODEL_ID
final_embeddings_df["final_text_variant"] = FINAL_TEXT_VARIANT_NAME
final_embeddings_df["final_similarity_metric"] = FINAL_SIMILARITY_METRIC
final_embeddings_df["final_selection_rule"] = FINAL_SELECTION_RULE
final_embeddings_df["final_locked_threshold"] = FINAL_LOCKED_THRESHOLD
final_embeddings_df["embedding_dim"] = int(normalized_embedding_matrix.shape[1])
final_embeddings_df["embedding_generation_seconds"] = float(embedding_elapsed_seconds)
final_embeddings_df["raw_embedding_norm"] = raw_embedding_norms.astype(float)
final_embeddings_df["normalized_embedding_norm"] = normalized_embedding_norms.astype(float)
final_embeddings_df["embedding_vector"] = [vec.tolist() for vec in normalized_embedding_matrix]

final_embeddings_df.to_parquet(FINAL_EMBEDDINGS_PARQUET_PATH, index=False)

print("Saved final embedding artifact to:", FINAL_EMBEDDINGS_PARQUET_PATH)
print("final_embeddings_df shape:", final_embeddings_df.shape)

display(final_embeddings_df.head(3))

Saved final embedding artifact to: C:\Users\user\Documents\thesis\data\final_embeddings.parquet
final_embeddings_df shape: (50, 14)


,course_unit_code,course_name,final_embedding_text,final_model_name,final_openai_model_id,final_text_variant,final_similarity_metric,final_selection_rule,final_locked_threshold,embedding_dim,embedding_generation_seconds,raw_embedding_norm,normalized_embedding_norm,embedding_vector
0,TT00CD70,Data Networks,Data Networks\nYou learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data ne...,openai_text-embedding-3-large,text-embedding-3-large,enriched,cosine,max_f1,0.673784,3072,1.445298,1.000048,1.0,"[-0.02810533344745636, 0.020735694095492363, -0.005145777948200703, 0.017394179478287697, 0.019438758492469788, -9.685286931926385e-05, -0.02070517651736736..."
1,TT00CD71,Linux Basics,"Linux Basics\nAfter completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work wit...",openai_text-embedding-3-large,text-embedding-3-large,enriched,cosine,max_f1,0.673784,3072,1.445298,0.999847,1.0,"[-0.013971555046737194, 0.03848854452371597, -0.012353876605629921, 0.0024856547825038433, -0.017489243298768997, 0.0045134760439395905, -0.0257302466779947..."
2,TT00CD72,Servers and containers,Servers and containers\nYou understand the fundamental types and functions of servers. You understand the key concepts of various services offered by cloud ...,openai_text-embedding-3-large,text-embedding-3-large,enriched,cosine,max_f1,0.673784,3072,1.445298,0.999609,1.0,"[-0.028056597337126732, 0.029399896040558815, -0.013387179933488369, 0.038039740175008774, 0.031140075996518135, -0.05470883846282959, -0.014768638648092747..."


In [19]:
# Cell 10 - Compute full pairwise similarity sutface

course_code_order = sorted(final_course_text_df["course_unit_code"].tolist())
course_name_map = dict(zip(final_course_text_df["course_unit_code"], final_course_text_df["course_name"]))
code_to_index = dict(zip(final_course_text_df["course_unit_code"], range(len(final_course_text_df))))

pair_records = []

for code_1, code_2 in combinations(course_code_order, 2):
    idx1 = code_to_index[code_1]
    idx2 = code_to_index[code_2]

    vec1 = normalized_embedding_matrix[idx1]
    vec2 = normalized_embedding_matrix[idx2]

    similarity_score = cosine_similarity_from_normalized(vec1, vec2)
    overlap_flag = int(similarity_score >= FINAL_LOCKED_THRESHOLD)

    pair_records.append({
        "pair_id": f"{code_1}__{code_2}",
        "course_unit_code_1": code_1,
        "course_name_1": course_name_map[code_1],
        "course_unit_code_2": code_2,
        "course_name_2": course_name_map[code_2],
        "similarity_score": float(similarity_score),
        "overlap_flag": int(overlap_flag),
        "final_locked_threshold": float(FINAL_LOCKED_THRESHOLD),
        "final_model_name": FINAL_MODEL_NAME,
        "final_text_variant": FINAL_TEXT_VARIANT_NAME,
        "final_similarity_metric": FINAL_SIMILARITY_METRIC,
        "final_selection_rule": FINAL_SELECTION_RULE,
    })

final_pairwise_similarity_df = pd.DataFrame(pair_records)

expected_pair_count = len(course_code_order) * (len(course_code_order) - 1) // 2
if len(final_pairwise_similarity_df) != expected_pair_count:
    raise ValueError(
        f"Expected {expected_pair_count} pair rows, got {len(final_pairwise_similarity_df)}."
    )

print("Expected pair count:", expected_pair_count)
print("Actual pair count:", len(final_pairwise_similarity_df))
display(final_pairwise_similarity_df.head(5))

Expected pair count: 1225
Actual pair count: 1225


,pair_id,course_unit_code_1,course_name_1,course_unit_code_2,course_name_2,similarity_score,overlap_flag,final_locked_threshold,final_model_name,final_text_variant,final_similarity_metric,final_selection_rule
0,TE00BR86__TE00CN56,TE00BR86,Back-End Development,TE00CN56,Internet Networks and Security,0.460966,0,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
1,TE00BR86__TE00CS88,TE00BR86,Back-End Development,TE00CS88,Introduction to Programming,0.501649,0,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
2,TE00BR86__TE00CS89,TE00BR86,Back-End Development,TE00CS89,Databases,0.549378,0,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
3,TE00BR86__TT00CD70,TE00BR86,Back-End Development,TT00CD70,Data Networks,0.442442,0,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
4,TE00BR86__TT00CD71,TE00BR86,Back-End Development,TT00CD71,Linux Basics,0.432365,0,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1


In [20]:
# Cell 11 - Save full pairwise similarity artifact

final_pairwise_similarity_df.to_parquet(FINAL_PAIRWISE_SIMILARITY_PARQUET_PATH, index=False)
final_pairwise_similarity_df.to_csv(FINAL_PAIRWISE_SIMILARITY_CSV_PATH, index=False, encoding="utf-8")

print("Saved pairwise similarity parquet to:", FINAL_PAIRWISE_SIMILARITY_PARQUET_PATH)
print("Saved pairwise similarity CSV to:", FINAL_PAIRWISE_SIMILARITY_CSV_PATH)

Saved pairwise similarity parquet to: C:\Users\user\Documents\thesis\data\final_pairwise_similarity.parquet
Saved pairwise similarity CSV to: C:\Users\user\Documents\thesis\data\final_pairwise_similarity.csv


In [21]:
# Cell 12 - Build final overlap candidate table

final_overlap_candidates_df = final_pairwise_similarity_df[
    final_pairwise_similarity_df["overlap_flag"] == 1
].copy()

final_overlap_candidates_df = final_overlap_candidates_df.sort_values(
    ["similarity_score", "pair_id"],
    ascending=[False, True]
).reset_index(drop=True)

final_overlap_candidates_df.to_csv(
    FINAL_OVERLAP_CANDIDATES_CSV_PATH,
    index=False,
    encoding="utf-8"
)

print("Saved final overlap candidates to:", FINAL_OVERLAP_CANDIDATES_CSV_PATH)
print("Number of above-threshold overlap candidates:", len(final_overlap_candidates_df))

display(final_overlap_candidates_df.head(20))

Saved final overlap candidates to: C:\Users\user\Documents\thesis\reports\tables\final_demo\final_demo_overlap_candidates.csv
Number of above-threshold overlap candidates: 31


,pair_id,course_unit_code_1,course_name_1,course_unit_code_2,course_name_2,similarity_score,overlap_flag,final_locked_threshold,final_model_name,final_text_variant,final_similarity_metric,final_selection_rule
0,TT00CD83__TT00DB42,TT00CD83,Data Structures and Algorithms,TT00DB42,Data Structures and Algorithms,0.845726,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
1,TT00CD92__TT00CD94,TT00CD92,Mobile Project,TT00CD94,Mobile Application Development,0.830978,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
2,TT00CD92__TT00DS75,TT00CD92,Mobile Project,TT00DS75,Mobile Application Development,0.788770,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
3,TT00CD93__TT00CD94,TT00CD93,Android Application Development,TT00CD94,Mobile Application Development,0.784151,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
4,TT00CD70__TT00CE20,TT00CD70,Data Networks,TT00CE20,Local Area Networks,0.783004,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
5,TT00CD94__TT00DS75,TT00CD94,Mobile Application Development,TT00DS75,Mobile Application Development,0.780172,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
6,TT00CE05__TT00CE06,TT00CE05,Data Analytics Project,TT00CE06,Machine Learning Project,0.778475,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
7,TT00CD91__TT00DS61,TT00CD91,Object Oriented Programming,TT00DS61,Object-Oriented Programming with Python,0.761706,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
8,TE00BR86__TT00CD85,TE00BR86,Back-End Development,TT00CD85,Backend Programming,0.758613,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
9,TT00CD87__TT00CE11,TT00CD87,Information Security Technologies,TT00CE11,Data Security Controls,0.747908,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1


In [22]:
# Cell 13 - Build top matches per course for demo inspection

top_match_rows = []

for source_code in course_code_order:
    left_df = final_pairwise_similarity_df[
        final_pairwise_similarity_df["course_unit_code_1"] == source_code
    ][[
        "course_unit_code_1",
        "course_name_1",
        "course_unit_code_2",
        "course_name_2",
        "similarity_score",
        "overlap_flag",
    ]].copy()

    left_df = left_df.rename(columns={
        "course_unit_code_1": "source_course_unit_code",
        "course_name_1": "source_course_name",
        "course_unit_code_2": "matched_course_unit_code",
        "course_name_2": "matched_course_name",
    })

    right_df = final_pairwise_similarity_df[
        final_pairwise_similarity_df["course_unit_code_2"] == source_code
    ][[
        "course_unit_code_2",
        "course_name_2",
        "course_unit_code_1",
        "course_name_1",
        "similarity_score",
        "overlap_flag",
    ]].copy()

    right_df = right_df.rename(columns={
        "course_unit_code_2": "source_course_unit_code",
        "course_name_2": "source_course_name",
        "course_unit_code_1": "matched_course_unit_code",
        "course_name_1": "matched_course_name",
    })

    source_matches_df = pd.concat([left_df, right_df], ignore_index=True)
    source_matches_df = source_matches_df.sort_values(
        ["similarity_score", "matched_course_unit_code"],
        ascending=[False, True]
    ).reset_index(drop=True)

    source_matches_df["match_rank"] = range(1, len(source_matches_df) + 1)
    source_matches_df["final_locked_threshold"] = FINAL_LOCKED_THRESHOLD

    top_match_rows.append(source_matches_df.head(10))

final_top_matches_df = pd.concat(top_match_rows, ignore_index=True)
final_top_matches_df.to_csv(FINAL_TOP_MATCHES_CSV_PATH, index=False, encoding="utf-8")

print("Saved top matches per course to:", FINAL_TOP_MATCHES_CSV_PATH)
print("final_top_matches_df shape:", final_top_matches_df.shape)

display(final_top_matches_df.head(20))

Saved top matches per course to: C:\Users\user\Documents\thesis\reports\tables\final_demo\final_demo_top_matches_per_course.csv
final_top_matches_df shape: (500, 8)


,source_course_unit_code,source_course_name,matched_course_unit_code,matched_course_name,similarity_score,overlap_flag,match_rank,final_locked_threshold
0,TE00BR86,Back-End Development,TT00CD85,Backend Programming,0.758613,1,1,0.673784
1,TE00BR86,Back-End Development,TT00DS73,Front-End Development,0.665820,0,2,0.673784
2,TE00BR86,Back-End Development,TT00CD78,Frontend Programming,0.615688,0,3,0.673784
3,TE00BR86,Back-End Development,TT00CD79,Basics of Web Development,0.589048,0,4,0.673784
4,TE00BR86,Back-End Development,TT00CD94,Mobile Application Development,0.557201,0,5,0.673784
5,TE00BR86,Back-End Development,TE00CS89,Databases,0.549378,0,6,0.673784
6,TE00BR86,Back-End Development,TT00CD80,JavaScript Programming,0.542713,0,7,0.673784
7,TE00BR86,Back-End Development,TT00CE17,Web Application Security Testing,0.522452,0,8,0.673784
8,TE00BR86,Back-End Development,TT00CD92,Mobile Project,0.515662,0,9,0.673784
9,TE00BR86,Back-End Development,TT00DS75,Mobile Application Development,0.514768,0,10,0.673784


In [23]:
# Cell 14 - Build final system summary

final_system_summary_df = pd.DataFrame([
    {"metric": "course_count", "value": int(len(final_course_text_df))},
    {"metric": "embedding_dimension", "value": int(normalized_embedding_matrix.shape[1])},
    {"metric": "pair_count", "value": int(len(final_pairwise_similarity_df))},
    {"metric": "above_threshold_pair_count", "value": int(final_pairwise_similarity_df["overlap_flag"].sum())},
    {"metric": "locked_threshold", "value": float(FINAL_LOCKED_THRESHOLD)},
    {"metric": "embedding_generation_seconds", "value": float(embedding_elapsed_seconds)},
    {"metric": "mean_similarity", "value": float(final_pairwise_similarity_df["similarity_score"].mean())},
    {"metric": "median_similarity", "value": float(final_pairwise_similarity_df["similarity_score"].median())},
    {"metric": "p95_similarity", "value": float(final_pairwise_similarity_df["similarity_score"].quantile(0.95))},
    {"metric": "max_similarity", "value": float(final_pairwise_similarity_df["similarity_score"].max())},
])

final_system_summary_df.to_csv(FINAL_SYSTEM_SUMMARY_CSV_PATH, index=False, encoding="utf-8")

print("Saved final system summary to:", FINAL_SYSTEM_SUMMARY_CSV_PATH)
display(final_system_summary_df)

Saved final system summary to: C:\Users\user\Documents\thesis\reports\tables\final_demo\final_demo_system_summary.csv


,metric,value
0,course_count,50.000000
1,embedding_dimension,3072.000000
2,pair_count,1225.000000
3,above_threshold_pair_count,31.000000
4,locked_threshold,0.673784
5,embedding_generation_seconds,1.445298
6,mean_similarity,0.457962
7,median_similarity,0.443649
8,p95_similarity,0.619446
9,max_similarity,0.845726


In [24]:
# Cell 15 - Save a one-row final configuration summary

final_configuration_df = pd.DataFrame([{
    "final_model_name": FINAL_MODEL_NAME,
    "final_openai_model_id": FINAL_OPENAI_MODEL_ID,
    "final_text_variant": FINAL_TEXT_VARIANT_NAME,
    "final_text_column": FINAL_TEXT_COLUMN,
    "final_similarity_metric": FINAL_SIMILARITY_METRIC,
    "final_selection_rule": FINAL_SELECTION_RULE,
    "final_locked_threshold": FINAL_LOCKED_THRESHOLD,
    "course_count": int(len(final_course_text_df)),
    "pair_count": int(len(final_pairwise_similarity_df)),
    "above_threshold_pair_count": int(final_pairwise_similarity_df["overlap_flag"].sum()),
}])

final_configuration_df.to_csv(FINAL_CONFIGURATION_CSV_PATH, index=False, encoding="utf-8")

print("Saved final configuration summary to:", FINAL_CONFIGURATION_CSV_PATH)
display(final_configuration_df)

Saved final configuration summary to: C:\Users\user\Documents\thesis\reports\tables\final_demo\final_demo_selected_configuration.csv


,final_model_name,final_openai_model_id,final_text_variant,final_text_column,final_similarity_metric,final_selection_rule,final_locked_threshold,course_count,pair_count,above_threshold_pair_count
0,openai_text-embedding-3-large,text-embedding-3-large,enriched,embedding_text_enriched,cosine,max_f1,0.673784,50,1225,31


In [25]:
# Cell 16 - Final integrity and app-readiness checks

artifact_integrity_summary = {
    "course_count_final_embeddings": len(final_embeddings_df),
    "unique_course_codes_final_embeddings": final_embeddings_df["course_unit_code"].nunique(),
    "empty_embedding_vector_rows": int(final_embeddings_df["embedding_vector"].isna().sum()),
    "pair_count_final_pairwise_similarity": len(final_pairwise_similarity_df),
    "unique_pair_ids_final_pairwise_similarity": final_pairwise_similarity_df["pair_id"].nunique(),
    "duplicate_pair_ids_final_pairwise_similarity": int(final_pairwise_similarity_df["pair_id"].duplicated().sum()),
    "top_matches_row_count": len(final_top_matches_df),
    "expected_top_matches_row_count": len(final_course_text_df) * 10,
    "overlap_candidate_count": len(final_overlap_candidates_df),
}

artifact_integrity_series = pd.Series(artifact_integrity_summary)
display(artifact_integrity_series)

if artifact_integrity_summary["course_count_final_embeddings"] != 50:
    raise ValueError("Final embedding artifact does not contain the expected 50 courses.")

if artifact_integrity_summary["unique_course_codes_final_embeddings"] != 50:
    raise ValueError("Final embedding artifact does not contain 50 unique course codes.")

if artifact_integrity_summary["empty_embedding_vector_rows"] > 0:
    raise ValueError("Some final embedding rows are missing embedding vectors.")

if artifact_integrity_summary["pair_count_final_pairwise_similarity"] != 1225:
    raise ValueError("Final pairwise similarity artifact does not contain 1225 pairs.")

if artifact_integrity_summary["duplicate_pair_ids_final_pairwise_similarity"] > 0:
    raise ValueError("Duplicate pair_id values found in final pairwise similarity artifact.")

if artifact_integrity_summary["top_matches_row_count"] != artifact_integrity_summary["expected_top_matches_row_count"]:
    raise ValueError("Top-match export row count is not equal to 50 * 10.")

course_count_final_embeddings                     50
unique_course_codes_final_embeddings              50
empty_embedding_vector_rows                        0
pair_count_final_pairwise_similarity            1225
unique_pair_ids_final_pairwise_similarity       1225
duplicate_pair_ids_final_pairwise_similarity       0
top_matches_row_count                            500
expected_top_matches_row_count                   500
overlap_candidate_count                           31
dtype: int64

In [26]:
# Cell 17 - Practical sanity checks for final demo inspection

print("Top 15 above-threshold overlap candidates:")
display(final_overlap_candidates_df.head(15))

print("\nTop 15 strongest course-to-course matches overall:")
display(
    final_pairwise_similarity_df
    .sort_values(["similarity_score", "pair_id"], ascending=[False, True])
    .head(15)
)

example_course_code = final_course_text_df.iloc[0]["course_unit_code"]
example_course_name = final_course_text_df.iloc[0]["course_name"]

print(f"\nTop matches for example course: {example_course_code} | {example_course_name}")
display(
    final_top_matches_df[
        final_top_matches_df["source_course_unit_code"] == example_course_code
    ].head(10)
)

Top 15 above-threshold overlap candidates:


,pair_id,course_unit_code_1,course_name_1,course_unit_code_2,course_name_2,similarity_score,overlap_flag,final_locked_threshold,final_model_name,final_text_variant,final_similarity_metric,final_selection_rule
0,TT00CD83__TT00DB42,TT00CD83,Data Structures and Algorithms,TT00DB42,Data Structures and Algorithms,0.845726,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
1,TT00CD92__TT00CD94,TT00CD92,Mobile Project,TT00CD94,Mobile Application Development,0.830978,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
2,TT00CD92__TT00DS75,TT00CD92,Mobile Project,TT00DS75,Mobile Application Development,0.788770,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
3,TT00CD93__TT00CD94,TT00CD93,Android Application Development,TT00CD94,Mobile Application Development,0.784151,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
4,TT00CD70__TT00CE20,TT00CD70,Data Networks,TT00CE20,Local Area Networks,0.783004,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
5,TT00CD94__TT00DS75,TT00CD94,Mobile Application Development,TT00DS75,Mobile Application Development,0.780172,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
6,TT00CE05__TT00CE06,TT00CE05,Data Analytics Project,TT00CE06,Machine Learning Project,0.778475,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
7,TT00CD91__TT00DS61,TT00CD91,Object Oriented Programming,TT00DS61,Object-Oriented Programming with Python,0.761706,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
8,TE00BR86__TT00CD85,TE00BR86,Back-End Development,TT00CD85,Backend Programming,0.758613,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
9,TT00CD87__TT00CE11,TT00CD87,Information Security Technologies,TT00CE11,Data Security Controls,0.747908,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1



Top 15 strongest course-to-course matches overall:


,pair_id,course_unit_code_1,course_name_1,course_unit_code_2,course_name_2,similarity_score,overlap_flag,final_locked_threshold,final_model_name,final_text_variant,final_similarity_metric,final_selection_rule
625,TT00CD83__TT00DB42,TT00CD83,Data Structures and Algorithms,TT00DB42,Data Structures and Algorithms,0.845726,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
848,TT00CD92__TT00CD94,TT00CD92,Mobile Project,TT00CD94,Mobile Application Development,0.830978,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
873,TT00CD92__TT00DS75,TT00CD92,Mobile Project,TT00DS75,Mobile Application Development,0.788770,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
874,TT00CD93__TT00CD94,TT00CD93,Android Application Development,TT00CD94,Mobile Application Development,0.784151,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
225,TT00CD70__TT00CE20,TT00CD70,Data Networks,TT00CE20,Local Area Networks,0.783004,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
924,TT00CD94__TT00DS75,TT00CD94,Mobile Application Development,TT00DS75,Mobile Application Development,0.780172,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
1072,TT00CE05__TT00CE06,TT00CE05,Data Analytics Project,TT00CE06,Machine Learning Project,0.778475,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
844,TT00CD91__TT00DS61,TT00CD91,Object Oriented Programming,TT00DS61,Object-Oriented Programming with Python,0.761706,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
15,TE00BR86__TT00CD85,TE00BR86,Back-End Development,TT00CD85,Backend Programming,0.758613,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1
745,TT00CD87__TT00CE11,TT00CD87,Information Security Technologies,TT00CE11,Data Security Controls,0.747908,1,0.673784,openai_text-embedding-3-large,enriched,cosine,max_f1



Top matches for example course: TT00CD70 | Data Networks


,source_course_unit_code,source_course_name,matched_course_unit_code,matched_course_name,similarity_score,overlap_flag,match_rank,final_locked_threshold
40,TT00CD70,Data Networks,TT00CE20,Local Area Networks,0.783004,1,1,0.673784
41,TT00CD70,Data Networks,TT00CE28,Core Networks,0.744962,1,2,0.673784
42,TT00CD70,Data Networks,TT00DS35,Introduction to Networks,0.718951,1,3,0.673784
43,TT00CD70,Data Networks,TE00CN56,Internet Networks and Security,0.706223,1,4,0.673784
44,TT00CD70,Data Networks,TT00CE21,Cloud Architectures and Platforms,0.592988,0,5,0.673784
45,TT00CD70,Data Networks,TT00CD86,Cyber Security,0.578102,0,6,0.673784
46,TT00CD70,Data Networks,TT00CE11,Data Security Controls,0.572231,0,7,0.673784
47,TT00CD70,Data Networks,TT00CD87,Information Security Technologies,0.564120,0,8,0.673784
48,TT00CD70,Data Networks,TT00CE25,Windows Infrastructure,0.561348,0,9,0.673784
49,TT00CD70,Data Networks,TT00CD72,Servers and containers,0.561306,0,10,0.673784


In [27]:
# Cell 18 - Course-level overlap density check

course_overlap_count_rows = []

for source_code in course_code_order:
    all_source_matches_left = final_pairwise_similarity_df[
        final_pairwise_similarity_df["course_unit_code_1"] == source_code
    ][["similarity_score", "overlap_flag"]].copy()

    all_source_matches_right = final_pairwise_similarity_df[
        final_pairwise_similarity_df["course_unit_code_2"] == source_code
    ][["similarity_score", "overlap_flag"]].copy()

    all_source_matches_df = pd.concat(
        [all_source_matches_left, all_source_matches_right],
        ignore_index=True
    )

    course_overlap_count_rows.append({
        "course_unit_code": source_code,
        "course_name": course_name_map[source_code],
        "above_threshold_match_count": int(all_source_matches_df["overlap_flag"].sum()),
        "best_similarity_score": float(all_source_matches_df["similarity_score"].max()),
        "mean_similarity_score": float(all_source_matches_df["similarity_score"].mean()),
    })

course_overlap_count_df = pd.DataFrame(course_overlap_count_rows).sort_values(
    ["above_threshold_match_count", "best_similarity_score", "course_unit_code"],
    ascending=[False, False, True]
).reset_index(drop=True)

display(course_overlap_count_df.head(20))

,course_unit_code,course_name,above_threshold_match_count,best_similarity_score,mean_similarity_score
0,TT00CD70,Data Networks,4,0.783004,0.485233
1,TT00CD87,Information Security Technologies,4,0.747908,0.493260
2,TT00CD92,Mobile Project,3,0.830978,0.470583
3,TT00CD94,Mobile Application Development,3,0.830978,0.490065
4,TT00DS75,Mobile Application Development,3,0.788770,0.455623
5,TT00CD93,Android Application Development,3,0.784151,0.473596
6,TT00CE20,Local Area Networks,2,0.783004,0.448224
7,TT00CE11,Data Security Controls,2,0.747908,0.491881
8,TT00CE28,Core Networks,2,0.744962,0.441480
9,TT00CD75,Windows Basics,2,0.738116,0.463836


In [28]:
# Cell 19 - Save a pgvector-ready export 

pgvector_ready_df = final_embeddings_df[[
    "course_unit_code",
    "course_name",
    "final_embedding_text",
    "final_model_name",
    "final_openai_model_id",
    "final_text_variant",
    "final_similarity_metric",
    "final_selection_rule",
    "final_locked_threshold",
    "embedding_dim",
    "embedding_vector",
]].copy()

pgvector_ready_df.to_parquet(PGVECTOR_READY_PARQUET_PATH, index=False)
pgvector_ready_df.to_csv(PGVECTOR_READY_CSV_PATH, index=False, encoding="utf-8")

print("Saved pgvector-ready parquet to:", PGVECTOR_READY_PARQUET_PATH)
print("Saved pgvector-ready CSV to:", PGVECTOR_READY_CSV_PATH)
print("pgvector_ready_df shape:", pgvector_ready_df.shape)

display(pgvector_ready_df.head(3))

Saved pgvector-ready parquet to: C:\Users\user\Documents\thesis\data\pgvector_ready_embeddings.parquet
Saved pgvector-ready CSV to: C:\Users\user\Documents\thesis\data\pgvector_ready_embeddings.csv
pgvector_ready_df shape: (50, 11)


,course_unit_code,course_name,final_embedding_text,final_model_name,final_openai_model_id,final_text_variant,final_similarity_metric,final_selection_rule,final_locked_threshold,embedding_dim,embedding_vector
0,TT00CD70,Data Networks,Data Networks\nYou learn the structure and protocols of data networks used by computers and other end devices. You know the principles of planning a data ne...,openai_text-embedding-3-large,text-embedding-3-large,enriched,cosine,max_f1,0.673784,3072,"[-0.02810533344745636, 0.020735694095492363, -0.005145777948200703, 0.017394179478287697, 0.019438758492469788, -9.685286931926385e-05, -0.02070517651736736..."
1,TT00CD71,Linux Basics,"Linux Basics\nAfter completing the course, you will understand the most important concepts of the Linux operating system (Linux OS) and know how to work wit...",openai_text-embedding-3-large,text-embedding-3-large,enriched,cosine,max_f1,0.673784,3072,"[-0.013971555046737194, 0.03848854452371597, -0.012353876605629921, 0.0024856547825038433, -0.017489243298768997, 0.0045134760439395905, -0.0257302466779947..."
2,TT00CD72,Servers and containers,Servers and containers\nYou understand the fundamental types and functions of servers. You understand the key concepts of various services offered by cloud ...,openai_text-embedding-3-large,text-embedding-3-large,enriched,cosine,max_f1,0.673784,3072,"[-0.028056597337126732, 0.029399896040558815, -0.013387179933488369, 0.038039740175008774, 0.031140075996518135, -0.05470883846282959, -0.014768638648092747..."


In [29]:
# Cell 20 - Final locked summary printout

print("Notebook 05 final status:")
print()
print("Final selected configuration")
print(f"  Model           : {FINAL_MODEL_NAME}")
print(f"  Text variant    : {FINAL_TEXT_VARIANT_NAME}")
print(f"  Similarity      : {FINAL_SIMILARITY_METRIC}")
print(f"  Selection rule  : {FINAL_SELECTION_RULE}")
print(f"  Locked threshold: {FINAL_LOCKED_THRESHOLD:.6f}")
print()
print("Final system outputs")
print(f"  Courses embedded              : {len(final_embeddings_df)}")
print(f"  Pairwise similarities computed: {len(final_pairwise_similarity_df)}")
print(f"  Above-threshold candidates    : {len(final_overlap_candidates_df)}")
print()
print("Artifacts")
print("  Final embedding artifact saved.")
print("  Final pairwise similarity artifact saved.")
print("  Final top-match table saved.")
print("  Final configuration summary saved.")
print("  pgvector-ready export saved.")
print()
print("Notebook 05 is complete and ready for the existing Streamlit demo.")

Notebook 05 final status:

Final selected configuration
  Model           : openai_text-embedding-3-large
  Text variant    : enriched
  Similarity      : cosine
  Selection rule  : max_f1
  Locked threshold: 0.673784

Final system outputs
  Courses embedded              : 50
  Pairwise similarities computed: 1225
  Above-threshold candidates    : 31

Artifacts
  Final embedding artifact saved.
  Final pairwise similarity artifact saved.
  Final top-match table saved.
  Final configuration summary saved.
  pgvector-ready export saved.

Notebook 05 is complete and ready for the existing Streamlit demo.


### Final note

Notebook 05 operationalizes the final selected overlap-detection configuration from Notebook 04 and prepares the reusable artifacts needed by the existing Streamlit proof-of-concept application.

### What this notebook does well

Key strengths of this notebook:

- applies the final selected configuration exactly as chosen in the earlier notebooks
- generates a complete embedding surface for all 50 courses
- produces the full 1225-pair similarity matrix for the corpus
- exports clean app-ready and pgvector-ready artifacts
- verifies that the final overlap-candidate set is internally consistent and practically inspectable

### Methodological role

Notebook 05 is not a new benchmark notebook.  
Its purpose is operational rather than comparative.

- Notebook 03 identified the strongest ranking configurations
- Notebook 04 selected and evaluated the final thresholded operating point
- Notebook 05 turns that selected configuration into a reusable end-to-end artifact set for the existing proof-of-concept application

### Practical interpretation

The output of this notebook shows that the selected configuration can be used as a stable proof-of-concept retrieval surface:

- all courses were embedded successfully
- the full pairwise similarity matrix was generated successfully
- the locked threshold produced a manageable candidate set
- the strongest retrieved pairs are interpretable and suitable for human inspection

This does not by itself guarantee production readiness.  
However, it does show that the selected model-threshold combination is operationally suitable for the thesis proof-of-concept and for the final demo application.

### Files produced by this notebook

Main artifacts:

- `data/final_embeddings.parquet`
- `data/final_pairwise_similarity.parquet`
- `data/final_pairwise_similarity.csv`
- `data/pgvector_ready_embeddings.parquet`
- `data/pgvector_ready_embeddings.csv`

Demo support tables:

- `reports/tables/final_demo/final_demo_selected_configuration.csv`
- `reports/tables/final_demo/final_demo_overlap_candidates.csv`
- `reports/tables/final_demo/final_demo_top_matches_per_course.csv`
- `reports/tables/final_demo/final_demo_system_summary.csv`

### Final role of Notebook 05

Notebook 05 does not replace the existing Streamlit application.  
Instead, it verifies and exports the final selected configuration so that the application can use a clean, validated, and reusable artifact set.